# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[Reena Aloraini]_ |
| **Student ID** | _[2220002781]_ |
| **Section** | _[FG01]_ |
| **Date submitted** | _[9 Oct, 2026]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [12]:
## I chose this dataset because I am gonna use it in another project later 
## it will be nice to explore it for this assignment first 

In [10]:
DATA_PATH = "../data/Reviews.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (568454, 10)
polars shape: (568454, 10)

columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']

   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought several of the Vitality canned d...  
1      Not as Advertised  Product arrived labeled as Jumbo Salted Peanut...  
2  "Deli

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [13]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [17]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.027 ms   <- nothing was executed
lazy WITH .collect():     359.611 ms   <- the real cost


In [15]:
## insane time difference

**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _[0.034, 0.026, 0.027]_ |
| Lazy query with `.collect()` | _[323.040, 376.478, 359.611]_ |

In one sentence — why is the first number meaningless?

_[because nothing was acutually excuted beside the line of code]_

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [18]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "ProductId"    # a text/category column, for grouping
NUMERIC_COL  = "Score"     # a numeric column
NUMERIC_COL2 = "HelpfulnessNumerator"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] ProductId
[ok] Score
[ok] HelpfulnessNumerator


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [20]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:  3496.82 ms
polars:   312.08 ms
ratio:     11.20x


**Observation:** _[polars is 11.2 to 11.37x times faster than pandas, I didn't expect this ]_

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [21]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 5.0
pandas:     0.91 ms
polars:     4.37 ms
ratio:      0.21x


**Observation:** _[Wow polars is actually slower here, maybe pandas is better in filtering, maybe the search mechanism in polars is worse]_

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [26]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     3.21 ms
polars:     4.95 ms
ratio:      0.65x


**Observation:** _[pandas:3.21, polars: 4.95 ms, not a huge difference, but again pandas here is faster, which supports my earlier thought about the search mechanism since we created a column from EXISTING columns that both libraries had to look for in the data   ]_

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [28]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:   129.94 ms
polars:    38.73 ms
ratio:      3.36x

pandas result (first rows):
ProductId
0006641040    161
141278509X      5
2734888454      7
2841233731      5
7310172001    822
Name: Score, dtype: int64

polars result (first rows):
shape: (5, 2)
┌────────────┬───────┐
│ ProductId  ┆ Score │
│ ---        ┆ ---   │
│ str        ┆ i64   │
╞════════════╪═══════╡
│ B00016AU3K ┆ 50    │
│ B001NZW2V6 ┆ 27    │
│ B0051C0Q3I ┆ 5     │
│ B002UUXL4K ┆ 9     │
│ B001EO6190 ┆ 39    │
└────────────┴───────┘


**Observation:** _[I noticed that pandas is slower than polars, but when I saw the results I understood why;sorts the groups by their keys, so the result is sorted alphabitically, whearase in polars, no sorting was involved in the grouping process, that's why it's faster ]_

**Do both libraries give the same numbers?** _[No]_

In [29]:
pandas_result = df_pandas.groupby("ProductId")["Score"].agg(
    ["sum", "mean"]
).reset_index()

print(pandas_result.head())

polars_result = df_polars.group_by("ProductId").agg(
    pl.col("Score").sum().alias("sum"),
    pl.col("Score").mean().alias("mean")
).sort("ProductId")

print(polars_result.head())

    ProductId  sum      mean
0  0006641040  161  4.351351
1  141278509X    5  5.000000
2  2734888454    7  3.500000
3  2841233731    5  5.000000
4  7310172001  822  4.751445
shape: (5, 3)
┌────────────┬─────┬──────────┐
│ ProductId  ┆ sum ┆ mean     │
│ ---        ┆ --- ┆ ---      │
│ str        ┆ i64 ┆ f64      │
╞════════════╪═════╪══════════╡
│ 0006641040 ┆ 161 ┆ 4.351351 │
│ 141278509X ┆ 5   ┆ 5.0      │
│ 2734888454 ┆ 7   ┆ 3.5      │
│ 2841233731 ┆ 5   ┆ 5.0      │
│ 7310172001 ┆ 822 ┆ 4.751445 │
└────────────┴─────┴──────────┘


In [31]:
## No they agreee :)
## because just taking the sum is meaningless in this case, we need the mean score

## 4.5 — Sorting and taking the top N

In [35]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:   146.12 ms
polars:    38.00 ms
ratio:      3.85x


**Observation:** _[pandas is slower here, even though I thought pandas is better at searching and sorting, the ~3.8x times difference is interesting]_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [37]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):  3525.03 ms
polars (lazy + collect):   272.29 ms
ratio:    12.95x


**Look at the query plan.** Polars can show you what it decided to do.

In [38]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("Score").sum()] BY [col("ProductId")]
  FROM
  Csv SCAN [../data/Reviews.csv]
  PROJECT 2/10 COLUMNS
  SELECTION: col("Score").cast(Float64) > 5.0
  ESTIMATED ROWS: 879838


**Observation:** _[what does the plan show? Can you see
anything in it about which columns are read, or where the filter is
applied? It is showing thr entire plan! but in a different order than the one in the code, I belive it optimizes the chain of operations by creating an order, and it shows which columns are read, and how many columns are in the data, and the filter "cast(Float64) > 5.0" ]_

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [40]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    93.59 ms
polars:    10.90 ms
ratio:      8.59x

rows before: 568454 -> after: 568454
row count unchanged: True


**Observation:** _[polars is faster than pandas in merging, and the number of rows remaind unchanged]_

---
# Part 5 · Results summary

Collect every measurement into one table.

In [41]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,3496.82,312.08,11.20,polars
1,filter,0.91,4.37,0.21,pandas
2,new column,3.21,4.95,0.65,pandas
3,group by,129.94,38.73,3.36,polars
4,sort + top N,146.12,38.00,3.85,polars
5,chained pipeline,3525.03,272.29,12.95,polars
6,join,93.59,10.90,8.59,polars


In [42]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.26300-SP0
   Intel64 Family 6 Model 154 Stepping 3, GenuineIntel


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

_[after putting all the results in one table, the differences became much clearer (I was rooting for pandas). The biggest differences were in the chained operations and file reading, in chained operation polars was 12x faster because we saw earlier how it optimizes the operation order first, unlike pandas where execution is in the order we specify. The second biggest difference was in file reading, the speedup was 11x, and I couldn't understand why it was this big even after running it a few times! I looked it up and appearently polars can use multiple CPU cores to read data :0 , I found it very interesting!]_

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

_[well the difference in times shoocked me a few times, some I was able to understand why, and some of them I had to look up. In the grouping and aggregating the results were different as first, because pandas preserve order (alphabitically), and polars does not, but when I took the mean of the scores, their results were the same]_

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

_[I can't answere this question without being biased, I have been dealing with pandas for two years, but I have never used polars, so dealing with pandas and understanding it was easier for me]_

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

_[the speedup was more than 5x in 3 out of 7 operations, any one can use similar operations to these 3 to make it seem like polars is better in a lot things. also things like the machine differences, and using different datasets.]_

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

_[Personally, if I had a project handed to me tommorow, I would choose pandas due to my familiarity, and in a lot of operations pandas was faster or very close to polars. However, if I was someone who was familar with both, and the project involved a lot of operations where polars is faster like needing to join a lot of tables, or needing an optimized chained operations, then yeah I would use polars]_

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

_[Dask, mainly used to handle very very large datasets because we can use parallelized versions of pandas DataFrames with it. It reminded me of tensors in PyTorch, where we split the data into tensors that get processed in parallel across many cores (num_workers). Documentation: https://docs.dask.org/en/stable/dataframe.html ]_

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot